# UNITREE G1 · GR00T 全身控制（WBC + finetune 路线）

**目标**：用 NVIDIA 的 **GEAR-SONIC 全身控制器（WBC）当平衡底座**，再 finetune GR00T 让 G1 跳舞 —— 平衡交给鲁棒 WBC，VLA 只出高层 latent，**from construction 不摔**（对比我们 raw-joint ACT 蒸馏 0.5s 必摔）。

## 为什么走这条路
- 我们的 ACT 学生直接出 29-DoF 关节角 → covariate shift → 闭环 0.5s 摔。
- `UNITREE_G1_SONIC` 路线：VLA → SONIC **latent** → GEAR-SONIC WBC 解码 → 稳定全身关节。WBC 已会走/跑/跳，VLA 不用学精细平衡。

## 关键事实（已核实）
- `UNITREE_G1` / `UNITREE_G1_SONIC` 是 GR00T 的**动作空间配置（embodiment tag）**，不是现成模型；是 **posttrain tag**，要 finetune 才有 checkpoint。
- **没有现成微调好的 G1 VLA checkpoint**（GR00T-WholeBodyControl 的 VLA workflow 明确要从 `nvidia/GR00T-N1.7-3B` base 自己 finetune）→ 所以我们 **WBC + finetune**。
- 真正可下载的就两个：
  | 模型 | 是什么 | 能否预览 G1 |
  |---|---|---|
  | `nvidia/GEAR-SONIC` | WBC 底座（walk/run/jump），encoder/decoder/planner ONNX | ✅ 能（键盘交互 MuJoCo viewer） |
  | `nvidia/GR00T-N1.7-3B` | VLA「大脑」base | ❌ 单独不能（要 +WBC +finetune） |

调研详见 `doc/vla_data_expansion_covariate_shift.html` §6（架构侧 SONIC WBC 路线）。

## ① GEAR-SONIC WBC — 下载 + 安装（一次性，较重）

clone `NVlabs/GR00T-WholeBodyControl` → 装 MuJoCo sim 依赖（需 TensorRT + C++ build）→ 下载 GEAR-SONIC checkpoints。
首次较慢；若 install 报错按 repo README 处理 TensorRT。

In [ ]:
!bash scripts/gear_sonic_setup.sh

## ① 实时预览 GEAR-SONIC 行为（Isaac Lab eval，会走/踢/舞的 base）

单进程 Isaac Sim viewer 跑已发布的 SONIC WBC（`sonic_release/last.pt`），让它实时**跟踪**参考动作 —— **无 DDS、无 C++**。下面 cell 先看一条 walk，快速确认环境通。

**viewer 快捷键**（先点一下视口让它获得焦点）：
- `F` 自由相机（**停止自动跟随**，可自由拉远/旋转看全场）
- `R` 重置所有 env（动作重头播） · `T` 下一批 motion · `V` 参考骨架（绿点=目标姿态，看跟踪误差）。

In [ ]:
!bash scripts/gear_sonic_preview.sh

## ③ 一键看 SONIC 各动作（架构侧 Milestone 0 go/no-go）

本地 deploy demo 动作转成 `robot_filtered` 后，让 SONIC WBC 实时**跟踪**。**跟得住不摔 = 该动作活在 SONIC 的 token 空间**，后面 VLA 只要吐对应 64 维 token 就能驱动它（详见 `doc/groot_sonic_wbc_route.html`）。

每个 cell = 一个 G1 跑一条动作。重点看 **kick / dance / macarena** 摔不摔、跟不跟得上参考。
> 首次运行任意一条会自动建 `data/demo_robot_filtered.pkl`（7 条一起转，之后秒起）。关闭 viewer 窗口或 `Ctrl+C` 结束。

### viewer 快捷键（先点一下视口让它获得焦点）
| 键 | 作用 |
|---|---|
| **`F`** | **自由相机** —— 停止自动跟随机器人，可自由拉远/旋转/缩放看全场（解决"拉近就被复位"） |
| `R` | 重置所有 env，动作从头播 |
| `T` | 切换下一批 motion |
| `V` | 开关参考骨架可视化（绿点 = 目标姿态，肉眼看跟踪误差） |

> 默认是"相机跟随 env 0"模式，所以你拖拽视角下一帧会被拽回 —— **按 `F` 即可解除**。

In [ ]:
# 🕺 跳舞 dance_in_da_party
!bash scripts/gear_sonic_demo.sh dance

In [ ]:
# 🕺 macarena
!bash scripts/gear_sonic_demo.sh macarena

In [ ]:
# 🥋 踢腿 neutral_kick
!bash scripts/gear_sonic_demo.sh kick

In [ ]:
# 🦵 弓步 forward_lunge
!bash scripts/gear_sonic_demo.sh lunge

In [ ]:
# 🦘 单腿跳 one_leg_jumping
!bash scripts/gear_sonic_demo.sh jump

In [ ]:
# 🏋️ 深蹲 squat
!bash scripts/gear_sonic_demo.sh squat

In [ ]:
# 🚶 转身走 walking_quip_360
!bash scripts/gear_sonic_demo.sh walk

### 全部 7 条一排（每个 G1 各跳一条，paired 排开）

In [ ]:
# 7 个 G1 一排，各跳一条
!bash scripts/gear_sonic_demo.sh

## ② GR00T-N1.7-3B base VLA — 下载（finetune 用）

下载 VLA「大脑」base。**单独不能驱动 G1**（要配 SONIC WBC + finetune）。
gated：可能需 `huggingface-cli login` + 申请 access。

In [ ]:
!bash scripts/groot_n17_download.sh

## WBC + finetune 路线图（下一步）

1. ✅ 下载 + 预览 GEAR-SONIC WBC（上面 ①）——确认它会把 G1 稳住。
2. **编码 latent 标签**：把我们的 dance/fight motion 经 SONIC **encoder**（`model_encoder.onnx`）转成 latent action 序列 → 这就是 VLA 的 action label。
3. **建 `UNITREE_G1_SONIC` LeRobot 数据集**：`(image, language, proprio) → SONIC latent`（动作空间换成 latent，不再是 29-DoF 关节）。
4. **finetune GR00T-N1.7**：用 Isaac-GR00T pipeline，`--base-model-path nvidia/GR00T-N1.7-3B` + `UNITREE_G1_SONIC` tag。
5. **部署闭环**：VLA → latent → SONIC **decoder** → G1，prompt "dance" 看是否**不摔地跳**。

> 数据侧的便宜替代（DART/DAgger 治 raw-joint ACT 的 covariate shift）见 `doc/vla_data_expansion_covariate_shift.html`。两条路互补：架构侧天花板高、架设重；数据侧便宜、快速见底。